# Task 04 Sobel Edge Detection using Cuda across many PNG images

Thaslim Mohammed Sahl | 2638120 | 6CS005

Runtime: **Python 3, T4 GPU, latest runtime version**. Use Runtime > Change runtime type to check this, then Runtime > Run all.

Each CUDA thread handles one pixel. The lecturer's rgbToGray logic uses 0.30R + 0.59G + 0.11B, calculated as (30R + 59G + 11B) / 100 with integer truncation. Gx measures changes from left to right and Gy from top to bottom, using two 3x3 kernels and zero padding. The original signed gradients give sqrt(Gx*Gx + Gy*Gy). Separate absolute X and Y gradient maps are saved as grayscale PNGs, while the final edge image preserves alpha. The demonstration uses the lecturer's single download.png input and shows its four views illustrated in the brief.

The assessment code is the separate `SobelEdge.cu` file, reproduced below using the lecturer's `%%writefile` workflow. Python is used for setup and independent verification. No Google Drive mount is needed.

Repository visibility is currently private at the student's request. Setup tries GitHub first and falls back to its compressed resource copy, with SHA-256 checks in both cases.

The brief illustrates four views of one input: Original Image, Gradient in X direction, Gradient in Y direction and Sobel Edge Detection. The normal demonstration processes only download.png and shows those four views. The CUDA command still accepts multiple PNGs as required by the brief; the separate regression cell generates small temporary inputs to check that capability and the gradient calculations.

The last-class sample stops at grayscale conversion. This source extends that taught PNG/CUDA workflow with the full Gx/Gy Sobel convolution required by Task 04. Its X/Y display buffers use one byte per pixel and LCT_GREY, 8 encoding, following the last-class example. Direct nvcc compilation is used; nvcc4jupyter is not needed for this %%writefile workflow.

Public demonstration input: [lecturer image, download.png](https://i.ibb.co/5gB80K0Z/download.png). Setup downloads this one image and checks its checksum and dimensions. Source details are in images/SOURCES.md. The isolated /content/HPC_Task_04_OneImage folder keeps earlier multi-image runs separate.

## Load the resources and check the environment

In [ ]:
# Resources are hosted with the project; a bundled copy supports private development.
from pathlib import Path
import os, io, base64, zipfile, hashlib, urllib.request, urllib.error, subprocess, sys
from urllib.parse import quote
ROOT = Path('/content/HPC_Task_04_OneImage')
ROOT.mkdir(parents=True, exist_ok=True)
RESOURCE_SHA256 = {'Practical Task/Task 04/lodepng.cpp': 'a7f82599ae6a4dbf0f3db3aa45a6125a1e88ba4d84c1f4be670bf3b4b5a27c86', 'Practical Task/Task 04/lodepng.h': '52a3e979fdc3048c804eec72a7927c3095ac254b03580f35539d46b72ff75f52', 'Practical Task/Task 04/images/download.png': 'bea2b0c28430f63466123f3db2d5f4b5d1a932aade1727d130bf318a490b6309', 'Practical Task/Task 04/images/public_images.json': '439abb79b4a1d76463f6a0fead9e565d41586c73a4f12fc25463293445fd9417', 'Practical Task/Task 04/images/SOURCES.md': 'dd735435b9533feca96c7d6498ab12a1ed2edc2c7122aff55ebf5d67bd3f70f3', 'tools/verify.py': '598e2b27efcd1fa7dcf5b9662d03e477d67aa24eaa97ae7e2260279f634428b9'}
RESOURCE_COPY = '...'
BASE_URL = 'https://raw.githubusercontent.com/sahl-2003/HPC/main/'
try:
    downloaded = {}
    for relative, digest in RESOURCE_SHA256.items():
        data = urllib.request.urlopen(BASE_URL + quote(relative), timeout=20).read()
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError('Resource checksum differs: ' + relative)
        downloaded[relative] = data
    for relative, data in downloaded.items():
        destination = ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
    print('Loaded and verified live GitHub resources.')
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, ValueError) as error:
    print('Using bundled resource copy for the current private repository:', type(error).__name__)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(RESOURCE_COPY))) as archive:
        for relative in RESOURCE_SHA256:
            data = archive.read(relative)
            assert hashlib.sha256(data).hexdigest() == RESOURCE_SHA256[relative]
            destination = ROOT / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(data)
os.chdir(ROOT / 'Practical Task' / 'Task 04')
print('Working folder:', Path.cwd())
print(subprocess.run(['gcc','--version'], capture_output=True, text=True, check=True).stdout.splitlines()[0])
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'], capture_output=True, text=True, check=True).stdout
print(gpu)
assert 'T4' in gpu, 'Select Runtime > Change runtime type > Python 3 > T4 GPU.'
print(subprocess.run(['nvcc','--version'], capture_output=True, text=True, check=True).stdout)
# Use wget, as in the last lecture, for the single public demonstration input.
import json
from PIL import Image
PUBLIC_IMAGES = json.loads(Path('images/public_images.json').read_text())
image_sources = []
for image in PUBLIC_IMAGES:
    destination = Path('images') / image['filename']
    temporary = destination.with_suffix('.download')
    result = subprocess.run(['wget', '--timeout=20', '--tries=2', '-q',
                             '-O', str(temporary), image['url']], capture_output=True, text=True)
    live_ok = (result.returncode == 0 and temporary.exists() and
               hashlib.sha256(temporary.read_bytes()).hexdigest() == image['sha256'])
    if live_ok:
        temporary.replace(destination)
        mode = 'live public download'
    else:
        temporary.unlink(missing_ok=True)
        assert hashlib.sha256(destination.read_bytes()).hexdigest() == image['sha256']
        mode = 'verified bundled copy (public source unavailable or changed)'
    with Image.open(destination) as png:
        assert png.format == 'PNG' and list(png.size) == image['dimensions']
        png.verify()
    print(image['filename'] + ': ' + mode)
    print('Public URL:', image['url'])
    image_sources.append(dict(image, load_mode=mode))
Path('image_downloads.json').write_text(json.dumps(image_sources, indent=2))
print('No Google Drive mount or GitHub login is needed for this image URL.')


## Write the separate C or CUDA source

In [ ]:
%%writefile SobelEdge.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <limits.h>
#include <math.h>
#include <time.h>
#include <sys/stat.h>
#include <cuda_runtime.h>
#include "lodepng.h"

__host__ __device__ unsigned char rgbToGray(const unsigned char *inImg, size_t i)
{
    /* The lecturer's 0.30, 0.59 and 0.11 weights, with exact integer truncation. */
    return (unsigned char)((30 * (int)inImg[i] + 59 * (int)inImg[i + 1] +
                            11 * (int)inImg[i + 2]) / 100);
}

__global__ void sobelEdge(const unsigned char *inImg, unsigned char *outImg,
                         unsigned char *gxImg, unsigned char *gyImg, int width, int height)
{
    size_t threadID = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (threadID >= (size_t)width * height) return;
    int x = (int)(threadID % width);
    int y = (int)(threadID / width);
    const int gxKernel[3][3] = {{-1, 0, 1}, {-2, 0, 2}, {-1, 0, 1}};
    const int gyKernel[3][3] = {{-1, -2, -1}, {0, 0, 0}, {1, 2, 1}};
    int gx = 0, gy = 0;
    for (int row = -1; row <= 1; row++) {
        for (int col = -1; col <= 1; col++) {
            int nx = x + col, ny = y + row;
            int value = 0; /* Zero padding outside the image. */
            if (nx >= 0 && nx < width && ny >= 0 && ny < height)
                value = rgbToGray(inImg, ((size_t)ny * width + nx) * 4);
            gx += value * gxKernel[row + 1][col + 1];
            gy += value * gyKernel[row + 1][col + 1];
        }
    }
    int absGx = gx < 0 ? -gx : gx;
    int absGy = gy < 0 ? -gy : gy;
    gxImg[threadID] = (unsigned char)(absGx > 255 ? 255 : absGx);
    gyImg[threadID] = (unsigned char)(absGy > 255 ? 255 : absGy);
    /* Combine signed gradients before clamping their separate display maps. */
    int magnitude = (int)sqrtf((float)(gx * gx + gy * gy));
    unsigned char edge = (unsigned char)(magnitude > 255 ? 255 : magnitude);
    size_t pixel = threadID * 4;
    outImg[pixel] = edge;
    outImg[pixel + 1] = edge;
    outImg[pixel + 2] = edge;
    outImg[pixel + 3] = inImg[pixel + 3];
}

/* A serial reference expands the two kernels into their neighbour sums. */
void sobelCPU(const unsigned char *input, unsigned char *output,
              unsigned char *gxOutput, unsigned char *gyOutput, int width, int height)
{
    for (int y = 0; y < height; y++) {
        for (int x = 0; x < width; x++) {
            int neighbourhood[3][3] = {{0}};
            for (int r = 0; r < 3; r++) for (int c = 0; c < 3; c++) {
                int nx = x + c - 1, ny = y + r - 1;
                if (nx >= 0 && nx < width && ny >= 0 && ny < height)
                    neighbourhood[r][c] = rgbToGray(input, ((size_t)ny * width + nx) * 4);
            }
            int gx = -neighbourhood[0][0] + neighbourhood[0][2]
                   - 2 * neighbourhood[1][0] + 2 * neighbourhood[1][2]
                   - neighbourhood[2][0] + neighbourhood[2][2];
            int gy = -neighbourhood[0][0] - 2 * neighbourhood[0][1] - neighbourhood[0][2]
                   + neighbourhood[2][0] + 2 * neighbourhood[2][1] + neighbourhood[2][2];
            size_t index = (size_t)y * width + x;
            int absGx = gx < 0 ? -gx : gx;
            int absGy = gy < 0 ? -gy : gy;
            gxOutput[index] = (unsigned char)(absGx > 255 ? 255 : absGx);
            gyOutput[index] = (unsigned char)(absGy > 255 ? 255 : absGy);
            int magnitude = (int)sqrtf((float)(gx * gx + gy * gy));
            size_t pixel = index * 4;
            output[pixel] = output[pixel + 1] = output[pixel + 2] =
                (unsigned char)(magnitude > 255 ? 255 : magnitude);
            output[pixel + 3] = input[pixel + 3];
        }
    }
}

int checkCuda(cudaError_t error, const char *action)
{
    if (error == cudaSuccess) return 1;
    fprintf(stderr, "CUDA %s: %s\n", action, cudaGetErrorString(error)); return 0;
}

double wallTime(void)
{
    struct timespec now;
    clock_gettime(CLOCK_MONOTONIC, &now);
    return now.tv_sec + now.tv_nsec / 1e9;
}

int processImage(const char *inputName, const char *outputName,
                 const char *gxName, const char *gyName)
{
    unsigned char *c_inImg = NULL, *c_outImg = NULL, *c_reference = NULL;
    unsigned char *c_gxImg = NULL, *c_gyImg = NULL;
    unsigned char *c_gxReference = NULL, *c_gyReference = NULL;
    unsigned char *d_inImg = NULL, *d_outImg = NULL;
    unsigned char *d_gxImg = NULL, *d_gyImg = NULL;
    unsigned int width = 0, height = 0;
    int status = 0;
    cudaEvent_t begin = NULL, finish = NULL;
    unsigned int pngError = lodepng_decode32_file(&c_inImg, &width, &height, inputName);
    if (pngError) {
        fprintf(stderr, "%s: PNG decode error %u: %s\n", inputName, pngError, lodepng_error_text(pngError));
        goto cleanup;
    }
    if (!width || !height || width > INT_MAX || height > INT_MAX ||
        (size_t)width > SIZE_MAX / 4 / height) {
        fprintf(stderr, "%s: unsupported image dimensions.\n", inputName); goto cleanup;
    }
    {
        size_t pixels = (size_t)width * height;
        size_t bytes = pixels * 4;
        cudaDeviceProp device;
        if (!checkCuda(cudaGetDeviceProperties(&device, 0), "device query")) goto cleanup;
        int threads = 256;
        if (threads > device.maxThreadsPerBlock) threads = device.maxThreadsPerBlock;
        size_t blocks = (pixels + threads - 1) / threads;
        if (blocks > (size_t)device.maxGridSize[0]) {
            fprintf(stderr, "%s: image exceeds the device grid limit.\n", inputName); goto cleanup;
        }
        c_outImg = (unsigned char *)malloc(bytes);
        c_reference = (unsigned char *)malloc(bytes);
        c_gxImg = (unsigned char *)malloc(pixels);
        c_gyImg = (unsigned char *)malloc(pixels);
        c_gxReference = (unsigned char *)malloc(pixels);
        c_gyReference = (unsigned char *)malloc(pixels);
        if (!c_outImg || !c_reference || !c_gxImg || !c_gyImg ||
            !c_gxReference || !c_gyReference) {
            fprintf(stderr, "Not enough host image memory.\n"); goto cleanup;
        }
        if (!checkCuda(cudaMalloc((void **)&d_inImg, bytes), "input allocation") ||
            !checkCuda(cudaMalloc((void **)&d_outImg, bytes), "output allocation") ||
            !checkCuda(cudaMalloc((void **)&d_gxImg, pixels), "Gx allocation") ||
            !checkCuda(cudaMalloc((void **)&d_gyImg, pixels), "Gy allocation") ||
            !checkCuda(cudaEventCreate(&begin), "start event") ||
            !checkCuda(cudaEventCreate(&finish), "end event")) goto cleanup;
        double gpuStart = wallTime();
        if (!checkCuda(cudaMemcpy(d_inImg, c_inImg, bytes, cudaMemcpyHostToDevice), "host to device") ||
            !checkCuda(cudaEventRecord(begin), "record start")) goto cleanup;
        sobelEdge<<<(unsigned int)blocks, threads>>>(d_inImg, d_outImg, d_gxImg, d_gyImg,
                                                   (int)width, (int)height);
        if (!checkCuda(cudaGetLastError(), "kernel launch") ||
            !checkCuda(cudaEventRecord(finish), "record end") ||
            !checkCuda(cudaDeviceSynchronize(), "kernel execution") ||
            !checkCuda(cudaMemcpy(c_outImg, d_outImg, bytes, cudaMemcpyDeviceToHost), "final device to host") ||
            !checkCuda(cudaMemcpy(c_gxImg, d_gxImg, pixels, cudaMemcpyDeviceToHost), "Gx device to host") ||
            !checkCuda(cudaMemcpy(c_gyImg, d_gyImg, pixels, cudaMemcpyDeviceToHost), "Gy device to host")) goto cleanup;
        double gpuMilliseconds = (wallTime() - gpuStart) * 1000;
        float kernelMilliseconds;
        if (!checkCuda(cudaEventElapsedTime(&kernelMilliseconds, begin, finish), "kernel timing")) goto cleanup;
        double cpuStart = wallTime();
        sobelCPU(c_inImg, c_reference, c_gxReference, c_gyReference, (int)width, (int)height);
        double cpuMilliseconds = (wallTime() - cpuStart) * 1000;
        if (memcmp(c_outImg, c_reference, bytes) || memcmp(c_gxImg, c_gxReference, pixels) ||
            memcmp(c_gyImg, c_gyReference, pixels)) {
            fprintf(stderr, "%s: GPU maps differ from serial CPU reference.\n", inputName); goto cleanup;
        }
        pngError = lodepng_encode32_file(outputName, c_outImg, width, height);
        if (pngError) {
            fprintf(stderr, "%s: PNG encode error %u: %s\n", outputName, pngError, lodepng_error_text(pngError));
            goto cleanup;
        }
        /* Each gradient map has one 8-bit grayscale sample per pixel. */
        pngError = lodepng_encode_file(gxName, c_gxImg, width, height, LCT_GREY, 8);
        if (pngError) {
            fprintf(stderr, "%s: PNG encode error %u: %s\n", gxName, pngError, lodepng_error_text(pngError));
            goto cleanup;
        }
        pngError = lodepng_encode_file(gyName, c_gyImg, width, height, LCT_GREY, 8);
        if (pngError) {
            fprintf(stderr, "%s: PNG encode error %u: %s\n", gyName, pngError, lodepng_error_text(pngError));
            goto cleanup;
        }
        printf("%s: %ux%u; %zu x blocks; %d threads/block\n", inputName, width, height, blocks, threads);
        printf("CPU compute: %.3f ms; CUDA kernel: %.3f ms; CUDA transfers + kernel: %.3f ms\n",
               cpuMilliseconds, kernelMilliseconds, gpuMilliseconds);
        printf("Gx, Gy and all final RGBA bytes match CPU reference: PASS\n");
        printf("Gradient in X: %s\nGradient in Y: %s\nOutput: %s\n\n", gxName, gyName, outputName);
        status = 1;
    }
cleanup:
    if (d_inImg && !checkCuda(cudaFree(d_inImg), "free input")) status = 0;
    if (d_outImg && !checkCuda(cudaFree(d_outImg), "free output")) status = 0;
    if (d_gxImg && !checkCuda(cudaFree(d_gxImg), "free Gx")) status = 0;
    if (d_gyImg && !checkCuda(cudaFree(d_gyImg), "free Gy")) status = 0;
    if (begin && !checkCuda(cudaEventDestroy(begin), "destroy start event")) status = 0;
    if (finish && !checkCuda(cudaEventDestroy(finish), "destroy end event")) status = 0;
    free(c_inImg); free(c_outImg); free(c_reference);
    free(c_gxImg); free(c_gyImg); free(c_gxReference); free(c_gyReference);
    return status;
}

const char *baseName(const char *path)
{
    const char *slash = strrchr(path, '/');
    return slash ? slash + 1 : path;
}

/* Every input generates final, Gx_ and Gy_ names. Reject collisions before writing. */
int outputNamesConflict(const char *first, const char *second)
{
    if (!strcmp(first, second)) return 1;
    const char *prefixes[2] = {"Gx_", "Gy_"};
    for (int i = 0; i < 2; i++) {
        if (!strncmp(first, prefixes[i], 3) && !strcmp(first + 3, second)) return 1;
        if (!strncmp(second, prefixes[i], 3) && !strcmp(second + 3, first)) return 1;
    }
    return 0;
}

int main(int argc, char *argv[])
{
    if (argc < 3) {
        fprintf(stderr, "Usage: %s output_directory image1.png [image2.png ...]\n", argv[0]); return 1;
    }
    struct stat directory;
    if (stat(argv[1], &directory) || !S_ISDIR(directory.st_mode)) {
        fprintf(stderr, "Output directory does not exist: %s\n", argv[1]); return 1;
    }
    /* Include cross-map names, such as foo.png and Gx_foo.png. */
    for (int i = 2; i < argc; i++) for (int j = 2; j < i; j++)
        if (outputNamesConflict(baseName(argv[i]), baseName(argv[j]))) {
            fprintf(stderr, "Input basenames would produce colliding output names.\n"); return 1;
        }
    cudaDeviceProp device;
    if (!checkCuda(cudaGetDeviceProperties(&device, 0), "device query")) return 1;
    printf("GPU: %s\n\n", device.name);
    int completed = 0;
    for (int i = 2; i < argc; i++) {
        const char *name = baseName(argv[i]);
        size_t capacity = strlen(argv[1]) + strlen(name) + 16;
        char *outputName = (char *)malloc(capacity);
        char *gxName = (char *)malloc(capacity);
        char *gyName = (char *)malloc(capacity);
        if (!outputName || !gxName || !gyName) {
            fprintf(stderr, "Filename allocation failed.\n");
            free(outputName); free(gxName); free(gyName); return 1;
        }
        snprintf(outputName, capacity, "%s/outImg_%s", argv[1], name);
        snprintf(gxName, capacity, "%s/outImg_Gx_%s", argv[1], name);
        snprintf(gyName, capacity, "%s/outImg_Gy_%s", argv[1], name);
        completed += processImage(argv[i], outputName, gxName, gyName);
        free(outputName); free(gxName); free(gyName);
    }
    printf("Successfully processed %d/%d PNG images.\n", completed, argc - 2);
    return completed == argc - 2 ? 0 : 1;
}


## Compile

In [ ]:
%%bash
set -e
nvcc -O2 -lineinfo -arch=sm_75 -Xcompiler=-Wall,-Wextra lodepng.cpp SobelEdge.cu -o SobelEdge

## Run the assessed program

In [ ]:
%%bash
set -e
mkdir -p outputs
./SobelEdge outputs images/download.png | tee task4_public_run.log
test ${PIPESTATUS[0]} -eq 0

## Display and check the output

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
name = 'download.png'
fig,axes = plt.subplots(2,2,figsize=(12,7))
views = [('Original Image', Path('images')/name),
         ('Gradient in X direction', Path('outputs')/('outImg_Gx_'+name)),
         ('Gradient in Y direction', Path('outputs')/('outImg_Gy_'+name)),
         ('Sobel Edge Detection', Path('outputs')/('outImg_'+name))]
for ax,(title,path) in zip(axes.flat,views):
    ax.imshow(Image.open(path).convert('RGBA'))
    ax.set_title(title); ax.axis('off')
fig.suptitle(name)
plt.tight_layout(); plt.show()
print('One input image is shown in four views. X/Y PNGs show min(255, abs(gradient)); the combined edge uses signed, unclipped Gx and Gy.')

## Independent correctness and failure tests

In [ ]:
test = subprocess.run([sys.executable, str(ROOT/'tools'/'verify.py'), '4'], text=True, capture_output=True)
print(test.stdout)
if test.stderr: print(test.stderr)
assert test.returncode == 0, 'Validation failed; inspect the error above.'

# Save the actual run outputs and validation evidence for the report.
from IPython.display import HTML, display
run_files = [Path('SobelEdge.cu'), Path('image_downloads.json'), Path('task4_public_run.log')]
run_files += sorted(Path('outputs').glob('*.png'))
run_files += sorted((ROOT/'evidence'/'validation').glob('task4_*'))
bundle = io.BytesIO()
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in run_files:
        archive.write(path, path.resolve().relative_to(ROOT).as_posix())
payload = base64.b64encode(bundle.getvalue()).decode()
display(HTML('<a download="Task_04_verified_outputs.zip" href="data:application/zip;base64,' + payload + '">Download Task 04 verified outputs (ZIP)</a>'))


## Viva notes

Each CUDA thread handles one pixel. The lecturer's rgbToGray logic uses 0.30R + 0.59G + 0.11B, calculated as (30R + 59G + 11B) / 100 with integer truncation. Gx measures changes from left to right and Gy from top to bottom, using two 3x3 kernels and zero padding. The original signed gradients give sqrt(Gx*Gx + Gy*Gy). Separate absolute X and Y gradient maps are saved as grayscale PNGs, while the final edge image preserves alpha. The demonstration uses the lecturer's single download.png input and shows its four views illustrated in the brief.

Be ready to explain allocation sizes, which values are shared, how the work is divided, and why the output is correct. Timing output distinguishes compute time from file operations and transfers. The notebook checks real outputs; it does not assume that extra threads always run faster.

[Task 04 notebook on Colab](https://colab.research.google.com/github/sahl-2003/HPC/blob/main/Practical%20Task/Task%2004/Task_04.ipynb). This link requires GitHub access while the repository is private.